In [1]:

# ============================================================
# RAILPULSE AI — INTERNAL TRAIN-LEVEL VALIDATION
# ============================================================

import os
import gc
import time
import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

DATA_PATH = "../data/processed/railpulse_model_features.csv"

OLD_MANIFEST_PATH = (
    "../outputs/tables/unseen_train_split_manifest.csv"
)

OUTPUT_DIR = "../outputs/tables"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CHUNK_SIZE = 100_000
RANDOM_STATE = 42

FEATURE_COLS = [
    "current_arr_delay",
    "distance_to_next",
    "current_distance",
    "current_sequence",
    "arrival_hour_sin",
    "arrival_hour_cos",
    "day_of_week_sin",
    "day_of_week_cos",
    "is_weekend"
]

TARGET_COL = "next_arr_delay"

USE_COLS = (
    ["train"] + FEATURE_COLS + [TARGET_COL]
)

# ------------------------------------------------------------
# 2. LOAD EXISTING TRAIN MANIFEST
# ------------------------------------------------------------

print("=" * 70)
print("RAILPULSE AI — INTERNAL VALIDATION EXPERIMENT")
print("=" * 70)

old_manifest = pd.read_csv(
    OLD_MANIFEST_PATH,
    dtype={"train": str}
)

development_trains = np.array(
    sorted(
        old_manifest.loc[
            old_manifest["split"] == "train",
            "train"
        ].astype(str).unique()
    )
)

final_test_trains = set(
    old_manifest.loc[
        old_manifest["split"] == "test",
        "train"
    ].astype(str)
)

print("\nDevelopment trains:", len(development_trains))
print("Reserved final-test trains:", len(final_test_trains))

# ------------------------------------------------------------
# 3. CREATE INTERNAL TRAIN / VALIDATION SPLIT
# ------------------------------------------------------------

rng = np.random.default_rng(RANDOM_STATE)

shuffled = rng.permutation(
    development_trains
)

validation_count = int(
    np.ceil(len(development_trains) * 0.20)
)

validation_trains = set(
    shuffled[:validation_count]
)

internal_train_trains = set(
    shuffled[validation_count:]
)

assert internal_train_trains.isdisjoint(
    validation_trains
)

assert internal_train_trains.isdisjoint(
    final_test_trains
)

assert validation_trains.isdisjoint(
    final_test_trains
)

print("\nINTERNAL SPLIT")
print("Training trains:", len(internal_train_trains))
print("Validation trains:", len(validation_trains))
print("Final test trains:", len(final_test_trains))

# Save complete three-way manifest
split_manifest = pd.DataFrame({
    "train": sorted(
        internal_train_trains |
        validation_trains |
        final_test_trains
    )
})

split_manifest["split"] = split_manifest[
    "train"
].map(
    lambda x:
        "internal_train" if x in internal_train_trains
        else "validation" if x in validation_trains
        else "reserved_final_test"
)

manifest_path = os.path.join(
    OUTPUT_DIR,
    "three_way_train_split_manifest.csv"
)

split_manifest.to_csv(
    manifest_path,
    index=False
)

print("\nSaved split manifest:", manifest_path)

# ------------------------------------------------------------
# 4. LOAD INTERNAL TRAINING AND VALIDATION DATA
# ------------------------------------------------------------

X_train_parts = []
y_train_parts = []

X_val_parts = []
y_val_parts = []

total_rows = 0

print("\nLoading data in chunks...")

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=USE_COLS,
    dtype={"train": str},
    chunksize=CHUNK_SIZE
):

    total_rows += len(chunk)

    train_mask = chunk["train"].isin(
        internal_train_trains
    ).to_numpy()

    val_mask = chunk["train"].isin(
        validation_trains
    ).to_numpy()

    X = chunk[FEATURE_COLS].to_numpy(
        dtype=np.float32
    )

    y = chunk[TARGET_COL].to_numpy(
        dtype=np.float32
    )

    if train_mask.any():
        X_train_parts.append(X[train_mask])
        y_train_parts.append(y[train_mask])

    if val_mask.any():
        X_val_parts.append(X[val_mask])
        y_val_parts.append(y[val_mask])

    if total_rows % 300_000 < CHUNK_SIZE:
        print(f"Processed {total_rows:,} rows")

X_train_val = np.concatenate(X_train_parts)
y_train_val = np.concatenate(y_train_parts)

X_validation = np.concatenate(X_val_parts)
y_validation = np.concatenate(y_val_parts)

del X_train_parts, y_train_parts
del X_val_parts, y_val_parts
gc.collect()

print("\nDATA LOADING COMPLETE")

print("Internal training shape:", X_train_val.shape)
print("Validation shape:", X_validation.shape)

print("Internal training rows:", f"{len(y_train_val):,}")
print("Validation rows:", f"{len(y_validation):,}")

assert len(X_train_val) + len(X_validation) <= total_rows

# ------------------------------------------------------------
# 5. PERSISTENCE BASELINE
# ------------------------------------------------------------

delay_index = FEATURE_COLS.index(
    "current_arr_delay"
)

current_train = X_train_val[:, delay_index]
current_val = X_validation[:, delay_index]

persistence_predictions = current_val.copy()

# ------------------------------------------------------------
# 6. DELTA LEARNING
# ------------------------------------------------------------

delta_target = (
    y_train_val - current_train
).astype(np.float32)

delta_model = HistGradientBoostingRegressor(
    loss="absolute_error",
    learning_rate=0.06,
    max_iter=100,
    max_leaf_nodes=31,
    min_samples_leaf=40,
    l2_regularization=2.0,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=12,
    random_state=RANDOM_STATE
)

print("\nTraining Delta Learning model...")

start = time.time()

delta_model.fit(
    X_train_val,
    delta_target
)

training_time = time.time() - start

predicted_change = delta_model.predict(
    X_validation
)

delta_predictions = np.maximum(
    current_val + predicted_change,
    0
)

# ------------------------------------------------------------
# 7. OVERALL EVALUATION
# ------------------------------------------------------------

def calculate_metrics(actual, predicted):

    return {
        "MAE": mean_absolute_error(
            actual, predicted
        ),
        "MSE": mean_squared_error(
            actual, predicted
        ),
        "RMSE": np.sqrt(
            mean_squared_error(actual, predicted)
        ),
        "R2": r2_score(
            actual, predicted
        )
    }

persistence_metrics = calculate_metrics(
    y_validation,
    persistence_predictions
)

delta_metrics = calculate_metrics(
    y_validation,
    delta_predictions
)

validation_results = pd.DataFrame([
    {
        "Model": "Persistence Baseline",
        **persistence_metrics,
        "Training_Time_Seconds": 0
    },
    {
        "Model": "Delta Learning",
        **delta_metrics,
        "Training_Time_Seconds": training_time
    }
])

print("\n" + "=" * 70)
print("INTERNAL VALIDATION RESULTS")
print("=" * 70)

print(
    validation_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

# ------------------------------------------------------------
# 8. CATEGORY-WISE ERROR ANALYSIS
# ------------------------------------------------------------

category_df = pd.DataFrame({
    "actual": y_validation,
    "persistence_prediction": persistence_predictions,
    "delta_prediction": delta_predictions
})

category_df["persistence_error"] = (
    category_df["actual"] -
    category_df["persistence_prediction"]
).abs()

category_df["delta_error"] = (
    category_df["actual"] -
    category_df["delta_prediction"]
).abs()

bins = [-0.001, 0, 15, 60, 180, 720, np.inf]

labels = [
    "On time (0)",
    "1-15 min",
    "16-60 min",
    "61-180 min",
    "181-720 min",
    ">720 min"
]

category_df["category"] = pd.cut(
    category_df["actual"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

category_results = category_df.groupby(
    "category",
    observed=False
).agg(
    observations=("actual", "size"),
    persistence_MAE=("persistence_error", "mean"),
    delta_MAE=("delta_error", "mean")
).reset_index()

category_results["MAE_improvement"] = (
    category_results["persistence_MAE"] -
    category_results["delta_MAE"]
)

print("\n" + "=" * 70)
print("CATEGORY-WISE VALIDATION RESULTS")
print("=" * 70)

print(
    category_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

# ------------------------------------------------------------
# 9. SAVE RESULTS
# ------------------------------------------------------------

validation_results.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "internal_validation_results.csv"
    ),
    index=False
)

category_results.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "internal_validation_category_results.csv"
    ),
    index=False
)

print("\nSaved validation results.")

# ------------------------------------------------------------
# 10. COMPLETION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("INTERNAL VALIDATION COMPLETED")
print("=" * 70)

print("Reserved final-test trains were not used for training.")
print("Train-level overlap checks passed.")
print("Review results before choosing the final model.")

# Release large temporary arrays after results are saved
del delta_model
del category_df
del delta_target
del predicted_change
del delta_predictions
del persistence_predictions

gc.collect()


RAILPULSE AI — INTERNAL VALIDATION EXPERIMENT

Development trains: 2516
Reserved final-test trains: 630

INTERNAL SPLIT
Training trains: 2012
Validation trains: 504
Final test trains: 630

Saved split manifest: ../outputs/tables\three_way_train_split_manifest.csv

Loading data in chunks...
Processed 300,000 rows
Processed 600,000 rows
Processed 900,000 rows

DATA LOADING COMPLETE
Internal training shape: (699293, 9)
Validation shape: (171336, 9)
Internal training rows: 699,293
Validation rows: 171,336

Training Delta Learning model...

INTERNAL VALIDATION RESULTS
               Model    MAE       MSE    RMSE     R2  Training_Time_Seconds
Persistence Baseline 9.2577 1003.8248 31.6832 0.8519                 0.0000
      Delta Learning 9.1626 1001.5029 31.6465 0.8522                10.5770

CATEGORY-WISE VALIDATION RESULTS
   category  observations  persistence_MAE  delta_MAE  MAE_improvement
On time (0)         56395           7.0672     6.8743           0.1928
   1-15 min         47486 

29